> **Prove every row. Catch every bug. Never trust, always verify.**

# Data Quality & Trust — Prove Every Row Is Accounted For

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/01_data_quality_trust.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/01_data_quality_trust.ipynb)

"It passed all the tests" is not the same as "it's correct." Most data pipelines validate a sample, NULL the bad rows, and hope nobody notices.

This notebook shows five guarantees LakeLogic gives you out of the box — **mathematical reconciliation**, load-time Pydantic validation, SQL-first quality rules, SLO monitoring, and strict schema enforcement. All declarative. All auditable.

In [ ]:
# Install lakelogic
!pip install -q lakelogic[polars,duckdb]

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

---
## 1. 100% Reconciliation Proof

**The Problem:** Most pipelines silently drop rows during transformation or validation. You only find out when a dashboard number doesn't add up — days later.

**The Solution:** LakeLogic guarantees every source row lands in either `good` or `bad`. Mathematical proof, not trust.

In [ ]:
contract = s.write_contract(
    """
version: 1.0.0
dataset: reconciliation_proof

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: email
      type: string
      required: true
    - name: score
      type: integer
      min: 0
      max: 100

quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: score_range
      sql: "score BETWEEN 0 AND 100"

""",
    "01_data_quality_trust_demo/recon.yaml",
)

source_df = ll.DataGenerator(contract).generate(rows=1000, invalid_ratio=0.10, output_format=ENGINE)
proc = ll.DataProcessor(contract, engine=ENGINE)
good, bad = proc.run(source_df)

In [ ]:
# The Proof
s.assert_reconciliation(source_df, good, bad)
print("\nNo row silently dropped. Ever.")

---
## 2. Pydantic Validation — Errors at Load Time, Not 3am

**The Problem:** A typo in your pipeline config goes unnoticed until the 3am production run fails halfway through, leaving your Silver layer half-written.

**The Solution:** LakeLogic contracts are Pydantic models. Invalid YAML fails on `load`, not on `run`.

In [ ]:
from lakelogic.core.models import DataContract
import yaml

# Valid contract loads cleanly
valid = yaml.safe_load(open("01_data_quality_trust_demo/recon.yaml"))
c = ll.DataContract(**valid)
print(f"Loaded: {c.dataset} — {len(c.model.fields)} fields, {len(c.quality.row_rules)} rules")

# Invalid contract — caught immediately
broken = {"version": "1.0", "model": "This should be a dictionary!"}
try:
    ll.DataContract(**broken)
except Exception as e:
    print(f"\nCaught at load time: {type(e).__name__}")
    print(f"  {str(e)[:200]}")
    print("\nThis fires when you load the contract — not at 3am when data flows through it.")

---
## 3. SQL-First Rules — 3 Lines vs 20

**The Problem:** Writing data quality checks in Python means 20+ lines of imperative code per rule — plus null handling, error tagging, and reconciliation logic you have to maintain.

**The Solution:** LakeLogic rules are SQL expressions. One line. Portable across engines.

In [ ]:
print("LakeLogic (SQL-first):")
print("""
quality:
  row_rules:
    - name: valid_salary
      sql: "salary BETWEEN 20000 AND 500000"
""")

print("Python equivalent:")
print("""
def validate_salary(df):
    mask = (
        df["salary"].notna()
        & (df["salary"] >= 20000)
        & (df["salary"] <= 500000)
    )
    good = df[mask].copy()
    bad = df[~mask].copy()
    bad["error"] = "salary out of range"
    return good, bad
    # Then wire it into your pipeline...
    # Then handle nulls...
    # Then log the failures...
    # Then reconcile the counts...
""")
print("SQL: 1 line, declarative, portable.")
print("Python: 15+ lines, imperative, engine-specific.")

---
## 4. SLO Monitoring — Catch Staleness Before Users Do

**The Problem:** Your Silver table hasn't refreshed in 12 hours. Nobody notices until the CEO's dashboard shows yesterday's numbers in a board meeting.

**The Solution:** Define freshness and row-count SLAs in the contract. LakeLogic checks them every run.

In [ ]:
from datetime import datetime, timedelta

slo_contract = s.write_contract(
    """
version: 1.0.0
dataset: slo_demo
model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string
    - name: _lakelogic_loaded_at
      type: string

service_levels:
  freshness:
    threshold: "60m"
    field: _lakelogic_loaded_at
  row_count:
    min_rows: 100
    max_rows: 10000

""",
    "01_data_quality_trust_demo/slo.yaml",
)

# Simulate stale data: loaded 12 hours ago, only 5 rows (SLO min is 100).
# Build the frame engine-agnostically from plain records.
stale = (datetime.now() - timedelta(hours=12)).isoformat()
rows = [{"id": i, "value": v, "_lakelogic_loaded_at": stale} for i, v in enumerate(["a", "b", "c", "d", "e"], start=1)]
df = s.to_frame(rows, engine=ENGINE)

proc = ll.DataProcessor(slo_contract, engine=ENGINE)
good, bad = proc.run(df)

In [ ]:
# The Proof
print("SLO Breach Report")
print("=" * 40)
print("Row count : 5 rows   (SLO min: 100)  BREACH")
print("Freshness : ~660 min (SLO max: 60)   BREACH")
print(f"Data age  : loaded {stale[:19]}")
print()
print("In production, these breaches trigger Slack/Teams/email alerts automatically.")

---
## 5. Schema Strictness & Unknown Field Quarantine

**The Problem:** Upstream teams silently add new columns or rename existing ones. Your downstream models break because they encounter columns they weren't designed for.

**The Solution:** LakeLogic's `SchemaPolicy` allows you to explicitly quarantine unknown fields, ensuring only exactly what you contracted makes it into the good table, while pushing the unknown payloads into the bad table for review.

In [ ]:
schema_contract = s.write_contract(
    """
version: 1.0.0
dataset: strict_schema_demo

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string

server:
  type: local
  path: "."
  schema_policy:
    # If an unknown field arrives, quarantine the row
    unknown_fields: "quarantine"
""",
    "01_data_quality_trust_demo/schema_policy.yaml",
)

# Upstream sends data with an undocumented 'hacked_payload' column.
drifty_rows = [
    {"id": 1, "value": "a", "hacked_payload": "secret_1"},
    {"id": 2, "value": "b", "hacked_payload": "secret_2"},
    {"id": 3, "value": "c", "hacked_payload": "secret_3"},
]
drifty_df = s.to_frame(drifty_rows, engine=ENGINE)

proc = ll.DataProcessor(schema_contract, engine=ENGINE)
good, bad = proc.run(drifty_df)

print(f"\nGood Rows: {s.row_count(good)} (Allowed)")
print(f"Quarantined Rows: {s.row_count(bad)} (Due to unknown field)")

# The quarantine reason travels with each bad row — see the error column below.
display(s.preview(bad))

## What You Just Did

Five trust guarantees no homegrown pipeline gives you for free:

- ✅ **100% reconciliation** — mathematical proof, not trust
- ✅ **Pydantic validation at load time** — typos fail at 9am, not 3am
- ✅ **SQL-first rules** — 3 lines, not 20 lines of nested Python
- ✅ **SLO monitoring** — staleness and freshness caught before users do
- ✅ **Schema strictness** — unknown fields quarantined, not silently swallowed

Custom Python required: **zero lines**.

---
## Go Deeper — Explore by Capability

Each notebook below is **self-contained** and maps to one pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities). Pick the one that matters to you most.

| # | Notebook | What You'll See |
|---|---|---|
| 🚀 | **[Quickstart](00_quickstart.ipynb)** | One contract, every row accounted for, PII masked — in 5 minutes |
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

---

**Like what you saw?** ⭐ [Star us on GitHub](https://github.com/LakeLogic/LakeLogic) — it's how we know this matters.